# 02 - EDA and validation of assumptions
Visual exploration, treatment of the `DAYS_EMPLOYED = 365243` placeholder, and a review of sensitive attributes.

In [ ]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from credit_risk.config import load_config
from credit_risk.data.loader import load_dataset, dataset_status_report
cfg = load_config()
print(dataset_status_report(cfg))
# demo=True is an EXPLICIT opt-in: if the real Home Credit files are missing, SYNTHETIC data is generated.
df, DATA_SOURCE = load_dataset(cfg, demo=True)
print("\nDATA SOURCE:", DATA_SOURCE, df.shape)
if DATA_SOURCE != "home_credit":
    print("WARNING: SYNTHETIC data - numbers are only a code-path demonstration, not real results.")

In [ ]:
from credit_risk.data.audit import make_eda_figures
fig_dir = ROOT / 'reports' / 'figures'
paths = make_eda_figures(df, fig_dir, 'TARGET', 'SK_ID_CURR')
print([p.name for p in paths])

In [ ]:
from IPython.display import Image, display
for p in paths[:4]: display(Image(filename=str(p)))

## Placeholder value in `DAYS_EMPLOYED`
`365243` is a placeholder (~1000 years) rather than a real tenure. It is converted to NaN and flagged by `DAYS_EMPLOYED_ANOM` in `FeatureEngineer`.

In [ ]:
m = df.DAYS_EMPLOYED == 365243
print('placeholder rows:', int(m.sum()), f'({m.mean():.1%})')
print('default rate placeholder vs. other:', round(df.loc[m,'TARGET'].mean(),4), round(df.loc[~m,'TARGET'].mean(),4))

## Sensitive attributes
Protected or proxy fields (`CODE_GENDER`, `DAYS_BIRTH` and age-derived ratios, `NAME_FAMILY_STATUS`) are excluded from the model by default (`features.exclude_sensitive`). Other fields can act as proxies (e.g. `ORGANIZATION_TYPE`, `OCCUPATION_TYPE`, `NAME_EDUCATION_TYPE`, region fields, family size) and are *not* removed; see `docs/methodology.md` for the limits of this approach. Below: observed default rate by these groups (descriptive only).

In [ ]:
for c in ['CODE_GENDER','NAME_FAMILY_STATUS']:
    print(df.groupby(c)['TARGET'].agg(['mean','size']).round(4), '\n')